# 🌍 Graph-RAG Travel Chatbot — LangChain Version

This is the **same chatbot** as `raw/GraphRAG_Travel_Chatbot_local.ipynb`, rebuilt with LangChain components instead of raw ChromaDB/transformers calls. Run this side-by-side with the original to see exactly what LangChain abstracts away versus what you still have to write yourself.

**What's identical:** the dummy data, the `networkx` graph, the Wikipedia-fetch + chunking idea, the graph-hop retrieval logic, the free local Hugging Face model, the three prompt templates, the keyword-based intent router.

**What's different (LangChain-ified):**

| Manual version | LangChain version |
|---|---|
| `chunk_text()` (word-window splitter) | `RecursiveCharacterTextSplitter` |
| `SentenceTransformer(...)` + manual `.encode()` calls | `HuggingFaceEmbeddings` |
| `chroma_client`, `collection.upsert/query` | `Chroma` vector store (`.from_documents`, `.similarity_search`) |
| Manual seed+hop function returning a string | A custom `GraphRAGRetriever(BaseRetriever)` class, pluggable into a chain |
| `pipeline("text-generation", ...)` called directly | `HuggingFacePipeline` + `ChatHuggingFace` wrapping the same pipeline |
| `TEMPLATE.format(context=..., query=...)` | `ChatPromptTemplate.from_messages([...])` |
| `travel_chatbot()` manually gluing steps together | An LCEL chain: `{"context": retriever \| format_docs, ...} \| prompt \| llm \| StrOutputParser()` |

**Still 100% custom, because LangChain doesn't provide this out of the box:** the graph-hop expansion logic itself (vector search seeds → walk `networkx` successors/predecessors → merge into context). LangChain has no built-in concept of *your* custom graph — we just wrap that same logic in LangChain's standard `BaseRetriever` interface so it can slot into a chain like any other retriever.

> ⚠️ **Version note:** LangChain's package layout changes fairly often (things move between `langchain`, `langchain_community`, `langchain_huggingface`, `langchain_chroma`). If an import below fails, it's almost always a version/package-split issue, not a logic error — check the error message for which package moved and adjust the `pip install` / `import` line accordingly.


## Step 0 — Install dependencies (Colab)


In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-chroma \
    chromadb sentence-transformers networkx transformers accelerate torch requests


## Step 1 — Dummy Data (identical to the manual notebook)

Same records: cities, attractions, transport options, categories.


In [ ]:
cities = [
    {"id": "paris", "name": "Paris", "country": "France",
     "description": "Paris is the capital of France, famous for art, fashion, romance, and the Eiffel Tower. Best visited Apr-Jun or Sep-Oct."},
    {"id": "tokyo", "name": "Tokyo", "country": "Japan",
     "description": "Tokyo is Japan's capital, blending ultra-modern skyscrapers with historic temples. Best visited Mar-May for cherry blossoms."},
    {"id": "goa", "name": "Goa", "country": "India",
     "description": "Goa is a coastal state in India known for beaches, nightlife, and Portuguese-era architecture. Best visited Nov-Feb."},
    {"id": "delhi", "name": "Delhi", "country": "India",
     "description": "Delhi is India's capital, home to Mughal-era monuments and a major international travel hub."},
]

attractions = [
    {"id": "eiffel_tower", "city_id": "paris", "name": "Eiffel Tower", "category": "landmark",
     "description": "Eiffel Tower: iconic iron tower in Paris, best visited at sunset, entry ~EUR 28, ~2 hours to explore."},
    {"id": "louvre", "city_id": "paris", "name": "Louvre Museum", "category": "culture",
     "description": "Louvre Museum: world's largest art museum in Paris, home to the Mona Lisa, entry ~EUR 22, allow half a day."},
    {"id": "senso_ji", "city_id": "tokyo", "name": "Senso-ji Temple", "category": "culture",
     "description": "Senso-ji: Tokyo's oldest Buddhist temple in Asakusa, free entry, best visited early morning to avoid crowds."},
    {"id": "shibuya", "city_id": "tokyo", "name": "Shibuya Crossing", "category": "landmark",
     "description": "Shibuya Crossing: world's busiest pedestrian crossing in Tokyo, free, best experienced at night."},
    {"id": "baga_beach", "city_id": "goa", "name": "Baga Beach", "category": "nature",
     "description": "Baga Beach: popular beach in North Goa known for water sports and beach shacks, free entry."},
    {"id": "fort_aguada", "city_id": "goa", "name": "Fort Aguada", "category": "landmark",
     "description": "Fort Aguada: 17th-century Portuguese fort in Goa with sea views, entry ~INR 25."},
    {"id": "red_fort", "city_id": "delhi", "name": "Red Fort", "category": "landmark",
     "description": "Red Fort: Mughal-era fortress in Delhi, UNESCO site, entry ~INR 35 for Indians / INR 550 for foreigners."},
]

transport_options = [
    {"id": "flight_del_par", "from_city": "delhi", "to_city": "paris", "mode": "flight",
     "cost_usd": 650, "duration_hours": 9,
     "description": "Flight from Delhi to Paris: ~9 hours, ~USD 650 round trip, most convenient option."},
    {"id": "flight_del_tokyo", "from_city": "delhi", "to_city": "tokyo", "mode": "flight",
     "cost_usd": 550, "duration_hours": 8,
     "description": "Flight from Delhi to Tokyo: ~8 hours, ~USD 550 round trip."},
    {"id": "flight_del_goa", "from_city": "delhi", "to_city": "goa", "mode": "flight",
     "cost_usd": 90, "duration_hours": 2.5,
     "description": "Flight from Delhi to Goa: ~2.5 hours, ~USD 90 round trip, fastest option."},
    {"id": "train_del_goa", "from_city": "delhi", "to_city": "goa", "mode": "train",
     "cost_usd": 35, "duration_hours": 26,
     "description": "Train from Delhi to Goa: ~26 hours, ~USD 35 one way, cheapest option, scenic but slow."},
    {"id": "bus_del_goa", "from_city": "delhi", "to_city": "goa", "mode": "bus",
     "cost_usd": 45, "duration_hours": 30,
     "description": "Bus from Delhi to Goa: ~30 hours, ~USD 45, budget option but long and tiring."},
]

categories = [
    {"id": "landmark", "name": "Landmark", "description": "Famous man-made structures and monuments."},
    {"id": "culture", "name": "Culture", "description": "Museums, temples, and cultural heritage sites."},
    {"id": "nature", "name": "Nature", "description": "Beaches, parks, and natural attractions."},
]

print(f"{len(cities)} cities, {len(attractions)} attractions, {len(transport_options)} transport options, {len(categories)} categories")


## Step 2 — Build the Knowledge Graph (plain `networkx`, no LangChain here)

LangChain doesn't have a concept of 'your custom graph' — this part stays 100% the same as the manual notebook.


In [ ]:
import networkx as nx

G = nx.DiGraph()

def add_node(node_id, node_type, **attrs):
    G.add_node(node_id, type=node_type, **attrs)

for c in cities:
    add_node(c["id"], "city", name=c["name"], description=c["description"])

for cat in categories:
    add_node(cat["id"], "category", name=cat["name"], description=cat["description"])

for a in attractions:
    add_node(a["id"], "attraction", name=a["name"], description=a["description"])
    G.add_edge(a["city_id"], a["id"], relation="HAS_ATTRACTION")
    G.add_edge(a["id"], a["category"], relation="HAS_CATEGORY")

for t in transport_options:
    add_node(t["id"], "transport", name=f"{t['mode'].title()} {t['from_city']}->{t['to_city']}",
             description=t["description"], cost_usd=t["cost_usd"], duration_hours=t["duration_hours"], mode=t["mode"])
    G.add_edge(t["from_city"], t["id"], relation="TRANSPORT_TO")
    G.add_edge(t["id"], t["to_city"], relation="TRANSPORT_TO")

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


## Step 2B — Real web content (Wikipedia) + Chunking via LangChain's `RecursiveCharacterTextSplitter`

Same idea as the manual notebook's Step 2B, but instead of a hand-rolled `chunk_text()` word-window function, we use LangChain's built-in splitter. `RecursiveCharacterTextSplitter` tries to split on paragraph breaks first, then sentences, then words — only falling back to a hard character cut if it has to. That generally produces cleaner chunk boundaries than a fixed word-count window.


In [ ]:
import requests

def fetch_wikipedia_extract(title, timeout=10):
    url = "https://en.wikipedia.org/w/api.php"
    params = {"action": "query", "prop": "extracts", "explaintext": True, "titles": title, "format": "json"}
    headers = {"User-Agent": "GraphRAGTravelChatbot/1.0 (educational Colab notebook)"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=timeout)
        resp.raise_for_status()
        pages = resp.json()["query"]["pages"]
        return next(iter(pages.values())).get("extract", "")
    except Exception as e:
        print(f"  [!] Failed to fetch '{title}': {e}")
        return ""


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# If the above import fails on your LangChain version, try instead:
# from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)  # sizes are in CHARACTERS here, not words

city_wiki_titles = {"paris": "Paris", "tokyo": "Tokyo", "goa": "Goa", "delhi": "Delhi"}
total_chunks_added = 0

for city_id, title in city_wiki_titles.items():
    article_text = fetch_wikipedia_extract(title)
    if not article_text:
        print(f"{title}: no content fetched, skipping")
        continue
    chunks = splitter.split_text(article_text)
    for idx, chunk_piece in enumerate(chunks):
        chunk_id = f"{city_id}_chunk_{idx}"
        add_node(chunk_id, "chunk", name=f"{title} (chunk {idx+1}/{len(chunks)})", description=chunk_piece)
        G.add_edge(city_id, chunk_id, relation="HAS_CHUNK")
    total_chunks_added += len(chunks)
    print(f"{title}: fetched {len(article_text)} chars -> {len(chunks)} chunks")

print(f"\nGraph now has {G.number_of_nodes()} nodes ({total_chunks_added} are Wikipedia chunks)")


## Step 3 — Embeddings + Vector Store, via LangChain

`HuggingFaceEmbeddings` wraps the exact same `sentence-transformers` model as before (`all-MiniLM-L6-v2`) behind LangChain's standard `Embeddings` interface — no `.encode()`/`.tolist()` calls needed, LangChain does that internally.

`Chroma.from_documents(...)` replaces the manual `chroma_client.get_or_create_collection(...)` + `collection.upsert(...)` pair — one call builds the collection *and* inserts everything. LangChain represents each item as a `Document` (`page_content` = the text that gets embedded, `metadata` = extra fields, exactly like Chroma's separate `documents`/`metadatas` lists before).


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
# If langchain_chroma isn't available on your version, try instead:
# from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

docs = []
for nid, data in G.nodes(data=True):
    text = f"[{data['type']}] {data.get('name','')}: {data.get('description','')}"
    docs.append(Document(page_content=text, metadata={"node_id": nid, "type": data["type"], "name": data.get("name", "")}))

# Rebuild the collection fresh each time this cell runs, so it always exactly matches the CURRENT graph G
# (avoids stale node ids left over from an earlier run if you re-ran Step 2/2B before this cell)
try:
    Chroma(embedding_function=embeddings_model, collection_name="travel_nodes_lc").delete_collection()
except Exception:
    pass  # nothing to delete yet on first run

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings_model,
    collection_name="travel_nodes_lc",
)

print(f"Inserted {len(docs)} node embeddings into the LangChain Chroma vector store")


## Step 4 — Custom Graph-RAG Retriever (LangChain `BaseRetriever` wrapping our own graph-hop logic)

This is the piece LangChain does **not** provide — we subclass `BaseRetriever` so our vector-search-then-graph-hop logic can plug into a chain like any built-in retriever, but the logic inside is identical to the manual notebook's `graph_rag_retrieve()`:
1. `vectorstore.similarity_search(query, k=top_k)` → seed nodes (LangChain's equivalent of `collection.query(...)`)
2. Walk each seed's graph successors/predecessors for `hops` rounds (plain `networkx`, unchanged)
3. Return the seeds + expanded nodes as a list of `Document` objects, which is the standard return type every LangChain retriever must produce


In [ ]:
from typing import List, Any
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from pydantic import ConfigDict

class GraphRAGRetriever(BaseRetriever):
    """LangChain-compatible retriever that does vector search + 1-hop graph expansion."""
    model_config = ConfigDict(arbitrary_types_allowed=True)

    vectorstore: Any
    graph: Any
    top_k: int = 4
    hops: int = 1

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        seed_docs = self.vectorstore.similarity_search(query, k=self.top_k)
        seed_ids = [d.metadata["node_id"] for d in seed_docs if d.metadata["node_id"] in self.graph]  # drop stale ids

        expanded_ids = set(seed_ids)
        frontier = set(seed_ids)
        for _ in range(self.hops):
            next_frontier = set()
            for nid in frontier:
                if nid in self.graph:
                    next_frontier.update(self.graph.successors(nid))
                    next_frontier.update(self.graph.predecessors(nid))
            expanded_ids.update(next_frontier)
            frontier = next_frontier

        results = []
        for nid in expanded_ids:
            if nid not in self.graph:  # belt-and-suspenders guard against stale vector-store entries
                continue
            data = self.graph.nodes[nid]
            tag = "SEED" if nid in seed_ids else "GRAPH-LINKED"
            content = f"[{tag}] ({data['type']}) {data.get('name','')}: {data.get('description','')}"
            results.append(Document(page_content=content, metadata={"node_id": nid, "type": data["type"], "tag": tag}))
        return results

retriever = GraphRAGRetriever(vectorstore=vectorstore, graph=G, top_k=4, hops=1)

# quick test
test_docs = retriever.invoke("cheapest way to reach the beach state from delhi")
for d in test_docs:
    print(d.page_content)


## Step 5 — LLM via LangChain (same free local Hugging Face model, wrapped)

`HuggingFacePipeline` wraps the exact same `transformers.pipeline(...)` object from the manual notebook. `ChatHuggingFace` sits on top of it so LangChain can send it structured chat messages (system/human roles) instead of one flat string — matching how `ChatPromptTemplate` (Step 6) formats things.


In [ ]:
from transformers import pipeline
import torch
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # same free, local, open-source model as the manual notebook
device = 0 if torch.cuda.is_available() else -1

hf_pipe = pipeline("text-generation", model=HF_MODEL, device=device, torch_dtype="auto", max_new_tokens=600)

llm = HuggingFacePipeline(pipeline=hf_pipe)
chat_model = ChatHuggingFace(llm=llm)

# --- Alternative: swap in a paid OpenAI-compatible API (e.g. Grok) with zero other changes ---
# from langchain_openai import ChatOpenAI
# chat_model = ChatOpenAI(model="grok-4.3", api_key="YOUR_KEY", base_url="https://api.x.ai/v1")


## Step 6 — Prompt Templates via `ChatPromptTemplate`

Same three templates as the manual notebook (itinerary / place-info / budget) and the same grounding system prompt — just expressed as a structured `ChatPromptTemplate` instead of an f-string `.format(...)` call.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = (
    "You are a helpful travel assistant. Answer ONLY using the CONTEXT provided below. "
    "If the context does not contain enough information, say so honestly instead of making facts up. "
    "Be concise, practical, and use bullet points where helpful."
)

itinerary_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "CONTEXT:\n{context}\n\nTASK: Create a day-by-day itinerary for the user's request below, "
              "using only attractions present in the context.\nUSER REQUEST: {query}"),
])

place_info_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "CONTEXT:\n{context}\n\nTASK: Answer the user's question about the place(s) using only the context above.\n"
              "USER REQUEST: {query}"),
])

budget_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "CONTEXT:\n{context}\n\nTASK: Compare the available transport options in the context in a small "
              "markdown table (mode, cost, duration), then recommend the best option for a budget traveler and the "
              "best for a time-constrained traveler.\nUSER REQUEST: {query}"),
])


## Step 7 — LCEL Chains + Intent Router

`format_docs` joins the retriever's `Document` list into one context string (same job as `"\n".join(context_lines)` in the manual notebook). Each chain is built with **LangChain Expression Language (LCEL)**: the `|` operator pipes the output of one step into the next.

`{"context": retriever | format_docs, "query": RunnablePassthrough()}` means: take the input string (the user's query), send it through `retriever` then `format_docs` to produce `context`, and *also* pass the original input straight through as `query` — both land as variables the prompt template expects.

The intent router itself is unchanged — LangChain doesn't replace this simple keyword logic, it's just Python.


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs) -> str:
    return "\n".join(d.page_content for d in docs)

def build_chain(prompt_template):
    return (
        {"context": retriever | format_docs, "query": RunnablePassthrough()}
        | prompt_template
        | chat_model
        | StrOutputParser()
    )

itinerary_chain = build_chain(itinerary_prompt)
place_info_chain = build_chain(place_info_prompt)
budget_chain = build_chain(budget_prompt)

def route_intent(query: str) -> str:
    q = query.lower()
    if any(w in q for w in ["itinerary", "plan", "day", "days", "trip plan", "schedule"]):
        return "itinerary"
    if any(w in q for w in ["cost", "budget", "cheap", "price", "flight", "train", "bus", "how to reach", "how to get", "travel option"]):
        return "budget"
    return "place_info"

CHAINS = {"itinerary": itinerary_chain, "budget": budget_chain, "place_info": place_info_chain}

def travel_chatbot(query: str, verbose: bool = False) -> str:
    intent = route_intent(query)
    if verbose:
        print(f"[intent detected: {intent}]")
    return CHAINS[intent].invoke(query)


## Step 8 — Request-Response Chat Loop

Identical UX to the manual notebook — plain `input()`/`print()`, no UI.


In [ ]:
def run_chat():
    print("Travel Chatbot (LangChain Graph-RAG demo). Type 'exit' to quit.\n")
    while True:
        user_query = input("You: ")
        if user_query.strip().lower() in {"exit", "quit"}:
            print("Bot: Safe travels! 👋")
            break
        answer = travel_chatbot(user_query, verbose=True)
        print(f"\nBot: {answer}\n")

# Uncomment to run interactively in Colab:
# run_chat()


## Step 9 — Sample Test Calls (no typing needed)


In [ ]:
print("=== ITINERARY ===")
print(travel_chatbot("Plan a 2 day itinerary for Paris", verbose=True))


In [ ]:
print("=== PLACE INFO ===")
print(travel_chatbot("Tell me about Senso-ji Temple in Tokyo", verbose=True))


In [ ]:
print("=== BUDGET COMPARISON ===")
print(travel_chatbot("What's the cheapest way to travel from Delhi to Goa?", verbose=True))


## Recap — Manual vs. LangChain, side by side

| Step | Manual notebook | This LangChain notebook |
|---|---|---|
| Data & Graph | hand-written dicts, `networkx` | **identical** — no LangChain equivalent exists for a custom graph |
| Chunking | `chunk_text()` word-window splitter | `RecursiveCharacterTextSplitter` (splits on paragraphs/sentences first) |
| Embeddings | `SentenceTransformer(...).encode(...)` | `HuggingFaceEmbeddings` (same underlying model, standard interface) |
| Vector store | `chromadb.Client()` + manual `upsert`/`query` | `Chroma.from_documents(...)` + `.similarity_search(...)` |
| Graph-hop retrieval | plain function returning a string | custom `BaseRetriever` subclass returning `Document`s — same logic, chain-pluggable |
| LLM | `transformers.pipeline(...)` called directly | `HuggingFacePipeline` + `ChatHuggingFace` (same pipeline, standard interface) |
| Prompts | f-string `.format(...)` | `ChatPromptTemplate.from_messages([...])` |
| Orchestration | one Python function calling each step | LCEL chain: `{...} \| prompt \| llm \| StrOutputParser()` |
| Intent routing | keyword `if/else` | **identical** — plain Python either way |

**Take-away:** LangChain didn't remove any of the RAG *concepts* you learned in the first notebook — retrieval, embeddings, chunking, prompting, generation are all still there, doing the same job. What it changed is *packaging*: standard interfaces (`Embeddings`, `BaseRetriever`, `Runnable`) that make it easy to swap one vector store, model, or prompt for another without touching the rest of the pipeline — which is exactly why the `# Alternative` comments in Steps 5 sit right there as one-line swaps instead of a rewrite.
